# Day 2 — Tool/Memory Validation, Multi-Hop Reasoning & Failure-Path Testing

**Module 6 · Agentic RAG Testing**

---

## What we'll cover today

| # | Topic | Why it matters |
|---|---|---|
| 1 | Why Module 5's RAGAS suite isn't enough here | Faithfulness/relevancy/precision/recall each have a specific blind spot once retrieval can loop |
| 2 | Memory validation across hops | Same boundary-value problem as Module 5's chunk-boundary bug, relocated to conversation memory |
| 3 | Hard negative: right facts, wrong combination | A failure mode per-fact faithfulness checks structurally cannot catch |
| 4 | Failure-path testing | What should happen when a hop comes up empty — the exact Klarna gap, testable |
| 5 | Extending the coverage matrix | New columns: `reasoning_chain_break`, `premature_stop`, `ungraceful_failure` |

**Estimated time:** 60 minutes
**Run order:** top to bottom. Several cells call the real agent from Day 1 -- make sure `.env` is configured in this `examples/` folder first (see Day 1's setup note).

---

> **Where we are in the course**
> Day 1 built a working, traced 2-hop loop and named 4 new failure modes that only exist in multi-step retrieval.
> Module 5 Day 2 taught boundary value analysis on chunk size; Module 5 Day 3 taught hard negatives for faithfulness.
> Today both come back, pointed at the planner's memory and its reasoning across hops instead of a single retrieval.

---
## Why Day 1's agent needs more than Module 5's test suite

Point Module 5's 4 RAGAS metrics at `agentic_rag()`'s final `response` and its accumulated `retrieved_contexts`, and three of them will happily score a *wrong* answer as fine:

- **`faithfulness`** asks "is every claim in the response backed by *some* retrieved chunk?" — it does not ask "was that the *right* chunk to base the answer on." Today's `reasoning_chain_break` hard negative cites WidgetPro 2000's real, retrieved, individually-true $50 fee — faithfulness has no way to know the question needed WidgetPro 3000's fee instead.
- **`answer_relevancy`** asks "does the response address the question asked?" — a confident, on-topic, wrong answer scores exactly the same as a confident, on-topic, right one.
- **`context_precision` / `context_recall`** ask about the *retrieved* chunks, not about what the generator did with them — they can see a retrieval error, but not a reasoning error layered on top of a clean retrieval.

None of this is a flaw in RAGAS. These metrics were built for a system where there's only ever one fact set to reason over — "did it combine multiple facts correctly" isn't a meaningful question when there's nothing to combine. It only becomes askable once retrieval can happen more than once, which is exactly what Day 1 added. Today builds the three checks that fill that gap:

1. **Memory validation** — did a fact from an early hop survive to the final answer, or get dropped along the way?
2. **Reasoning-combination checks** — were the *right* facts picked and combined, not just retrieved?
3. **Graceful-failure checks** — did the agent admit a gap instead of guessing, when a hop genuinely came up empty?

None of these three have a Module 5 equivalent — not because Module 5's authors missed them, but because none of them could occur in a system that only ever retrieves once.


---
## Memory validation: did the early fact survive?

A multi-hop chain passes facts from early hops forward into the final generation step. Module 4 Day 4 taught you to boundary-test a context window; this is the same boundary, relocated: **does the fact from hop 1 still make it into the final answer, or does it get dropped or corrupted by the time hop 3 generates a response?**

> **Plain English:** this is the chunk-boundary bug from Module 5 Day 2, except the "chunk" is the running memory of a multi-hop conversation instead of a document. The boundary is wherever your agent truncates history — and it can split a needed fact out exactly the same way a 500-token chunk boundary could.

In [1]:
import json
from pydantic import BaseModel
from agent import agentic_rag, judge_client, judge_model

# Load all golden cases once — keyed by id, reused across all 3 checks below.
with open("golden_dataset.json") as f:
    _golden = {c["id"]: c for c in json.load(f)}


In [2]:
class MemoryRetentionVerdict(BaseModel):
    retained: bool
    reasoning: str


async def judge_memory_retention(question: str, must_include: list[str], answer: str) -> MemoryRetentionVerdict:
    prompt = (
        f"Question: {question}\n\n"
        "A correct answer must explicitly reference all of the following facts from earlier retrieval hops:\n"
        + "\n".join(f"- {f}" for f in must_include)
        + f"\n\nAnswer to evaluate: {answer}\n\n"
        "Did the answer retain all required facts? A memory failure is when the answer gives correct "
        "information but drops the specific entity it belongs to — e.g. the right fee but no mention "
        "of which product it applies to. Respond with retained=true only if every required fact appears."
    )
    return await judge_client.chat.completions.create(
        model=judge_model,
        response_model=MemoryRetentionVerdict,
        messages=[{"role": "user", "content": prompt}],
    )


# Real agent answer — positive case, live model call.
memory_case = _golden["multi-hop-widgetpro-01-memory-drop"]
day1_result = await agentic_rag(memory_case["user_input"])
answer_with_memory = day1_result.response

# Hard negative from golden_dataset.json — hop 1's product identity dropped.
answer_memory_dropped = memory_case["response"]

for label, answer in [("REAL AGENT (memory intact)", answer_with_memory),
                       ("HARD NEGATIVE (memory dropped)", answer_memory_dropped)]:
    verdict = await judge_memory_retention(memory_case["user_input"], memory_case["must_include"], answer)
    print(f"[{label}] -> retained={verdict.retained}")
    print(f"    reasoning: {verdict.reasoning}")
    print(f"    answer: {answer!r}")

print()
print(f"day1_result.hit_max_hops = {day1_result.hit_max_hops}  (num_hops={day1_result.num_hops})")
print("False here means the planner confirmed it had enough — worth checking whenever")
print("a memory-validation case unexpectedly fails, since True would explain a lot on its own.")


[REAL AGENT (memory intact)] -> retained=False
    reasoning: The required fact is 'WidgetPro 3000,' which is the specific product that replaced WidgetPro 2000. The answer states the cancellation fee is $0 but does not mention WidgetPro 3000 at all, only referring to 'the product that replaced WidgetPro 2000.' This fails to explicitly include the required entity.
    answer: 'The cancellation fee for the product that replaced WidgetPro 2000 is $0.'
[HARD NEGATIVE (memory dropped)] -> retained=False
    reasoning: The correct answer states the cancellation fee ($0) but fails to mention the product this fee applies to, which is 'WidgetPro 3000' as required. Since not all required facts from the retrieval hops are explicitly referenced in the answer, this is a memory failure.
    answer: 'The cancellation fee is $0.'

day1_result.hit_max_hops = False  (num_hops=2)
False here means the planner confirmed it had enough — worth checking whenever
a memory-validation case unexpectedly fails, si

---
## Hard negative — right facts, wrong combination (`reasoning_chain_break`)

This is the failure mode Module 5's single-hop metrics structurally cannot catch, because both retrieved facts are individually faithful to their source — the bug is in how they were *combined*.

In [3]:
class ReasoningChainVerdict(BaseModel):
    correct_combination: bool
    reasoning: str


async def judge_reasoning_chain(
    question: str, must_include: list[str], must_not_include: list[str], answer: str
) -> ReasoningChainVerdict:
    prompt = (
        f"Question: {question}\n\n"
        f"A correct answer must include all of: {must_include}\n"
        f"A correct answer must NOT include any of: {must_not_include}\n\n"
        f"Answer to evaluate: {answer}\n\n"
        "Did the agent correctly combine the retrieved facts and reference the right entity? "
        "Both retrieved facts may be individually true — the failure is applying facts from the "
        "wrong entity (e.g. using WidgetPro 2000's fee to answer a question about its replacement). "
        "Respond with correct_combination=true only if must_include is satisfied AND must_not_include is absent."
    )
    return await judge_client.chat.completions.create(
        model=judge_model,
        response_model=ReasoningChainVerdict,
        messages=[{"role": "user", "content": prompt}],
    )


# Hard negative from golden_dataset.json — right facts, wrong combination.
reasoning_neg = _golden["multi-hop-widgetpro-01-hardneg"]

# Positive case — reuse the real agent answer from the memory check above (same question).
for label, answer in [("HARD NEGATIVE (should fail)", reasoning_neg["response"]),
                       ("REAL AGENT (should pass)", answer_with_memory)]:
    verdict = await judge_reasoning_chain(
        reasoning_neg["user_input"],
        reasoning_neg["must_include"],
        reasoning_neg["must_not_include"],
        answer,
    )
    print(f"[{label}] -> correct_combination={verdict.correct_combination}")
    print(f"    reasoning: {verdict.reasoning}")
    print(f"    answer: {answer!r}")

print()
print("A faithfulness check alone would PASS the hard negative above — '$50' really is in the")
print("retrieved context. The LLM judge catches the combination error because it reasons about")
print("which entity the question asked about, not just whether each fact is grounded.")


[HARD NEGATIVE (should fail)] -> correct_combination=False
    reasoning: The answer states the fee is $50, which is in the must_not_include list. The answer fails to include the required elements from must_include: it mentions the replacement product only implicitly ('the product that replaced WidgetPro 2000') without naming it as 'WidgetPro 3000', and it does not state the correct fee '$0'. Therefore, the combination of retrieved facts is incorrect.
    answer: 'The cancellation fee for the product that replaced WidgetPro 2000 is $50.'
[REAL AGENT (should pass)] -> correct_combination=True
    reasoning: The answer includes 'WidgetPro 3000' and '$0' (satisfying must_include) and does not include '$50' (satisfying must_not_include). The agent correctly identified that the product replacing WidgetPro 2000 is WidgetPro 3000 and applied the correct cancellation fee for that replacement product, not the fee for WidgetPro 2000 itself.
    answer: 'The cancellation fee for the product that 

---
## Failure-path testing: the hop that comes up empty

What should the agent do when a hop finds nothing relevant?

- **Graceful** — admits the gap: *"I found that WidgetPro 3000 replaced WidgetPro 2000, but I don't have its cancellation fee on file."*
- **Ungraceful** — confidently invents a number to fill the gap.

This hard negative is aimed squarely at the failure pattern behind Klarna's "complex cases dropped in quality" from Day 1 — an agent that can't find the next fact should say so, not fabricate one to keep the chain moving.

In [4]:
class GracefulFailureVerdict(BaseModel):
    is_graceful: bool
    reasoning: str


async def judge_graceful_failure(question: str, answer: str) -> GracefulFailureVerdict:
    prompt = (
        f"Question: {question}\n\n"
        "Context: The knowledge base contains no information that can answer this question.\n\n"
        f"Answer to evaluate: {answer}\n\n"
        "Did the agent handle the knowledge gap gracefully? "
        "Graceful means the agent explicitly admits it cannot find or does not have the answer — "
        "any phrasing that communicates 'the facts don't cover this' counts. "
        "Ungraceful means the agent invents or fabricates a specific fact (a number, date, policy) "
        "that was not in the knowledge base. Respond with is_graceful=true only for admissions of the gap."
    )
    return await judge_client.chat.completions.create(
        model=judge_model,
        response_model=GracefulFailureVerdict,
        messages=[{"role": "user", "content": prompt}],
    )


# Hard negative from golden_dataset.json — fabricated answer for a question with no corpus answer.
graceful_neg = _golden["graceful-failure-01-hardneg"]

# Real agent on the same question — corpus genuinely has no cancellation fee for TurboMax Pro.
missing_fact_result = await agentic_rag(graceful_neg["user_input"], verbose=True)
real_agent_answer = missing_fact_result.response

for label, answer in [("REAL AGENT", real_agent_answer),
                       ("HARD NEGATIVE (fabricated)", graceful_neg["response"])]:
    verdict = await judge_graceful_failure(graceful_neg["user_input"], answer)
    print(f"[{label}] -> is_graceful={verdict.is_graceful}")
    print(f"    reasoning: {verdict.reasoning}")
    print(f"    answer: {answer!r}")

print()
print(f"missing_fact_result.hit_max_hops = {missing_fact_result.hit_max_hops}")
print("This SHOULD be True — the corpus has no answer, so the planner should never confirm enough_info.")
print("If it's False, the planner hallucinated confidence — worth investigating separately.")
print()
print("If REAL AGENT came back is_graceful=False, that IS the finding this check exists to surface —")
print("it means the generator needs a stronger hedge instruction, not that the notebook is broken.")


[hop 1] query='What is the cancellation fee for TurboMax Pro?'
[hop 1] retrieved: ["TurboMax Pro's annual subscription costs $299, unchanged from TurboMax 5's price.", "WidgetPro 3000's cancellation fee is $0 -- it can be canceled anytime at no charge."]
[hop 1] planner.enough_info=False  reasoning="The retrieved facts give cancellation fee information for WidgetPro 3000 ($0), but no information on the cancellation fee for TurboMax Pro. The question specifically asks about TurboMax Pro's cancellation fee, so we need a fact directly about that product."
[hop 1] next_query='TurboMax Pro cancellation fee policy'

[hop 2] query='TurboMax Pro cancellation fee policy'
[hop 2] retrieved: ["TurboMax Pro's annual subscription costs $299, unchanged from TurboMax 5's price.", 'TurboMax 5 was renamed to TurboMax Pro in 2024 after a rebranding update.']
[hop 2] planner.enough_info=False  reasoning="We have information about TurboMax Pro's pricing, rebranding history, and even cancellation fees for 

---
## Extending the coverage matrix

Module 5 Day 2 added retrieval columns to Module 4 Day 4's matrix. Today adds the agentic ones -- and they map directly onto this notebook's opening argument: `reasoning_chain_break` is the check for the failure faithfulness can't see, `ungraceful_failure` is the check for the failure no RAGAS metric asks about at all.


In [5]:
import json
from collections import Counter

with open("golden_dataset.json") as f:
    golden_cases = json.load(f)

categories    = sorted({c["category"] for c in golden_cases})
failure_modes = sorted({c["failure_mode"] for c in golden_cases})
counts        = Counter((c["category"], c["failure_mode"]) for c in golden_cases)

header = " " * 16 + "".join(f"{fm:<24}" for fm in failure_modes)
print(header)
for cat in categories:
    row = f"{cat:<16}" + "".join(f"{counts[(cat, fm)]:<24}" for fm in failure_modes)
    print(row)

zero_cells = [(cat, fm) for cat in categories for fm in failure_modes if counts[(cat, fm)] == 0]
print()
if zero_cells:
    print("Still-empty cells (not necessarily a problem -- just visible now):")
    for cat, fm in zero_cells:
        print(f"- {cat} x {fm}")
else:
    print("No empty cells for these categories x failure modes -- Day 1's named premature_stop gap")
    print("is filled (see multi-hop-turbomax-01 in golden_dataset.json). single_hop_qa correctly")
    print("has no agentic-failure rows -- those columns only apply once a question needs multiple hops.")


                hallucination           premature_stop          reasoning_chain_break   ungraceful_failure      
multi_hop_qa    2                       2                       3                       2                       
single_hop_qa   2                       0                       0                       0                       

Still-empty cells (not necessarily a problem -- just visible now):
- single_hop_qa x premature_stop
- single_hop_qa x reasoning_chain_break
- single_hop_qa x ungraceful_failure


---
## Try It Yourself

1. Write a third memory-validation case where the final answer reflects hop 1's fact but **drops hop 2's entirely**. Does `facts_present_in_answer()` catch it the same way it caught the fully-dropped case above?
2. Add a `query_drift` row to `golden_dataset.json` -- a case where the reformulated query in hop 2 wanders away from the original question's intent (`query_drift` is named in Day 1's failure-mode table but has no row yet). What would its `retrieved_contexts` need to look like for a hard negative to actually demonstrate the drift, rather than just a wrong answer?
3. Write your own "right facts, wrong combination" hard negative in a domain other than product fees (e.g. dates, locations, prices) and add it to `golden_dataset.json` with `eval_type: "reasoning"`, `must_include`, and `must_not_include` fields (see `multi-hop-widgetpro-01` for the pattern). What made it easy or hard to construct compared to the WidgetPro example?

Exercise file: [`exercises/02_tool_memory_reasoning_exercise.md`](../exercises/02_tool_memory_reasoning_exercise.md)


---
## Summary

### What we built today
- A memory-validation check that catches facts dropped between hops — run against the REAL agent's answer, not a scripted stand-in — the conversational version of Module 5's chunk-boundary bug
- A `reasoning_chain_break` hard negative that a per-fact faithfulness check would have missed entirely
- A graceful-vs-ungraceful failure check aimed at the exact gap behind the Klarna incident
- A coverage matrix extended with `reasoning_chain_break`, `premature_stop`, and `ungraceful_failure`

### The thread through this whole module
Every check built across these two days reused a Module 4/5 technique and pointed it one level up: equivalence partitioning -> hop count, boundary value analysis -> conversation memory, hard negatives -> reasoning combination and graceful failure, coverage matrix -> agentic failure modes. Same mindset, new surface — same habit this course keeps building.

**Next:** Module 7 — AI Agents Testing with DeepEval, where these hand-rolled checks become formal, reusable metrics (task completion, tool correctness, argument correctness) and the agent gains real tool-calling, not just retrieval.

---